# E19 — O agora

O capítulo anterior mediu o corte no espaço: onde a série começa e termina. Este mede o corte no
**tempo**: há um instante em que a decisão é tomada, e os dados em que ela se apoia são de antes
disso. A distância entre os dois é o atraso, e quase ninguém o declara.

O atraso entra no corte do primeiro capítulo de um jeito simples: a barreira que vale hoje é a que
foi calculada com o dado de **atraso** dias atrás. Com atraso zero, isso é exatamente o corte do
primeiro capítulo — a identidade é conferida pelo auto_teste da biblioteca.

A pergunta do caderno é quanto ele custa. A hipótese, que se mede, é que o atraso não custa nada
num mundo que não muda --- o corte é um quantil, e o quantil não sabe que dia é hoje --- e que o que
ele cobra é a mudança que coube dentro dele.


In [1]:
# <- brinque com: SERIE, ATRASOS, JANELA, CAUDA, SEMENTES, QUANDO, FATOR, HORIZONTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, mudanca, promessa, volatilidade

SERIE = "sp500.csv"
ATRASOS = (0, 1, 5, 21, 63, 252)
JANELA = 252
CAUDA = 0.05
POSTO = 13
PROMESSA = POSTO / (JANELA + 1)
SEMENTES = 20
DIAS_POR_MUNDO = 12000
QUANDO = 6000
FATOR = 2.0
HORIZONTE = 250

retornos = volatilidade.retornos_log(dados.carregar_serie(SERIE))
print("%s: %d dias | a promessa do corte e %.5f" % (SERIE, retornos.size, PROMESSA))


sp500.csv: 6718 dias | a promessa do corte e 0.05138


In [2]:
# O dado real: a entrega do corte contra o atraso.
linhas = []
for atraso in ATRASOS:
    v = promessa.violacoes_atrasadas(retornos, JANELA, CAUDA, atraso)
    linhas.append({"atraso": atraso, "entrega": float(v.mean()), "dias": int(v.size),
                   "razao": float(v.mean() / PROMESSA)})
real = pd.DataFrame(linhas).set_index("atraso")
print(real.round(5).to_string())


        entrega  dias    razao
atraso                        
0       0.05135  6466  0.99926
1       0.05182  6465  1.00845
5       0.05231  6461  1.01811
21      0.05400  6445  1.05083
63      0.05716  6403  1.11244
252     0.06003  6214  1.16819


In [3]:
# O mundo parado e o mundo que dobra: o atraso custa o que a mudanca couber nele.
linhas = []
for atraso in ATRASOS:
    parado, mudado, depois = [], [], []
    for i in range(SEMENTES):
        x = mudanca.degrau(DIAS_POR_MUNDO, np.random.default_rng(600 + i), fator=1.0, quando=QUANDO)
        v = promessa.violacoes_atrasadas(pd.Series(x), JANELA, CAUDA, atraso)
        parado.append(float(v.mean()))
        y = pd.Series(mudanca.degrau(DIAS_POR_MUNDO, np.random.default_rng(600 + i), fator=FATOR, quando=QUANDO))
        w = promessa.violacoes_atrasadas(y, JANELA, CAUDA, atraso)
        mudado.append(float(w.mean()))
        depois.append(float(w.loc[QUANDO:QUANDO + HORIZONTE].mean()))
    linhas.append({"atraso": atraso, "parado": float(np.mean(parado)),
                   "mudado": float(np.mean(mudado)), "depois_da_mudanca": float(np.mean(depois))})
mundos = pd.DataFrame(linhas).set_index("atraso")
print(mundos.round(5).to_string())
print()
print("no mundo parado o atraso de %d dias muda a entrega em %+.5f"
      % (ATRASOS[-1], mundos.loc[ATRASOS[-1], "parado"] - mundos.loc[0, "parado"]))


         parado   mudado  depois_da_mudanca
atraso                                     
0       0.05113  0.05243            0.11235
1       0.05115  0.05248            0.11315
5       0.05113  0.05252            0.11554
21      0.05119  0.05281            0.12689
63      0.05112  0.05334            0.15299
252     0.05156  0.05623            0.21116

no mundo parado o atraso de 252 dias muda a entrega em +0.00043


In [4]:
# Figura 1: a entrega contra o atraso, nas tres situacoes.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
posicoes = np.arange(len(ATRASOS))
eixo.plot(posicoes, real["entrega"], marker="o", color="#b03a2e", lw=1.6, label="o dado real")
eixo.plot(posicoes, mundos["parado"], marker="s", color="#1f4e79", lw=1.6, label="mundo parado")
eixo.plot(posicoes, mundos["depois_da_mudanca"], marker="^", color="#2e7d32", lw=1.6,
          label="mundo que dobra, na janela depois da mudança")
eixo.axhline(PROMESSA, color="#555555", ls="--", lw=1.4, label="a promessa do corte")
eixo.set_xticks(posicoes)
eixo.set_xticklabels([str(a) for a in ATRASOS], fontsize=9)
eixo.set_xlabel("atraso do dado, em dias --- a escala não é proporcional")
eixo.set_ylabel("fração dos dias que romperam o corte")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E19_o_agora", 1)
plt.close(fig)
print("reaL: %s" % [round(v, 5) for v in real["entrega"]])
print("parado: %s" % [round(v, 5) for v in mundos["parado"]])


reaL: [0.05135, 0.05182, 0.05231, 0.054, 0.05716, 0.06003]
parado: [0.05113, 0.05115, 0.05113, 0.05119, 0.05112, 0.05156]


In [5]:
# Figura 2: o preco do atraso, medido so na janela que segue a mudanca.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
largura = 0.35
posicoes = np.arange(len(ATRASOS))
eixo.bar(posicoes - largura / 2, mundos["parado"], largura, color="#1f4e79", label="mundo parado")
eixo.bar(posicoes + largura / 2, mundos["depois_da_mudanca"], largura, color="#b03a2e",
         label="mundo que dobra, depois da mudança")
eixo.axhline(PROMESSA, color="#555555", ls="--", lw=1.4, label="a promessa do corte")
eixo.set_xticks(posicoes)
eixo.set_xticklabels([str(a) for a in ATRASOS], fontsize=9)
eixo.set_xlabel("atraso do dado, em dias")
eixo.set_ylabel("fração dos dias que romperam o corte")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E19_o_agora", 2)
plt.close(fig)
print("depois da mudanca: %s" % [round(v, 4) for v in mundos["depois_da_mudanca"]])


depois da mudanca: [0.1124, 0.1131, 0.1155, 0.1269, 0.153, 0.2112]


## Leitura visual das figuras

Figura 1. Três curvas contra o atraso do dado, mais a linha tracejada da promessa. O dado real e o
mundo parado andam praticamente colados na linha da promessa em toda a extensão: são dois traços
que se confundem com a reta tracejada, e só no atraso maior o dado real se descola um pouco para
cima, sem nunca chegar perto da terceira curva. Essa terceira, o mundo que dobra com a janela
depois da mudança, já começa bem acima das outras no atraso zero, anda quase plana pelos atrasos
pequenos e sobe com força no último ponto, encostando no topo da moldura. O eixo horizontal é
logarítmico, com marcas em zero, um, cinco, 21, 63 e 252, e é aí que ele engana: no logaritmo o
intervalo de cinco a 21 dias ocupa o mesmo espaço que o de 63 a 252, de modo que o pulo que de
fato importa fica comprimido no canto direito e parece apenas mais um passo do mesmo tamanho. O
eixo vertical engana pelo lado oposto: a curva de cima obriga a escala a subir, as três linhas de
baixo ficam empilhadas na base, e a distância entre elas, que é o que o capítulo mede, some dentro
da espessura do traço.

Figura 2. Seis pares de barras, um par por atraso, com a barra azul do mundo parado e a vermelha
do mundo que dobra depois da mudança, e a linha tracejada da promessa atravessando o gráfico. As
azuis formam uma faixa reta no nível exato da linha tracejada: em nenhum atraso a barra azul se
distingue da promessa, e é isso que a figura diz sobre o mundo que não muda. As vermelhas sobem a
cada passo, passando do dobro da azul já no atraso zero e terminando quase no topo da moldura, com
o eixo começando em zero, de modo que a razão entre as alturas é honesta. O que engana é o
espaçamento horizontal: as seis categorias ficam igualmente separadas no papel, e os atrasos que
elas nomeiam não são igualmente separados no tempo, porque o salto de 21 para 63 dias tem a mesma
largura do salto de zero para um dia. A figura sugere um crescimento regular, passo a passo, que
ela mesma não pode sustentar.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {0: "zero", 1: "um", 5: "cinco", 21: "vinte_e_um", 63: "sessenta_e_tres", 252: "duzentos_e_cinquenta_e_dois"}
resultado = {
    "agora_dias": int(retornos.size),
    "agora_promessa": float(PROMESSA),
    "agora_janela": int(JANELA),
    "agora_atraso_maior": int(ATRASOS[-1]),
    "agora_sementes": int(SEMENTES),
    "agora_horizonte": int(HORIZONTE),
    "agora_parado_variacao": float(mundos.loc[ATRASOS[-1], "parado"] - mundos.loc[0, "parado"]),
    "agora_parado_menor": float(mundos["parado"].min()),
    "agora_parado_maior": float(mundos["parado"].max()),
}
for atraso in ATRASOS:
    nome = NOMES[atraso]
    resultado["agora_entrega_%s" % nome] = float(real.loc[atraso, "entrega"])
    resultado["agora_razao_%s" % nome] = float(real.loc[atraso, "razao"])
    if atraso in (0, 21):
        resultado["agora_parado_%s" % nome] = float(mundos.loc[atraso, "parado"])
    if atraso in (0, ATRASOS[-1]):
        resultado["agora_depois_%s" % nome] = float(mundos.loc[atraso, "depois_da_mudanca"])

caminho = Path("lab/resultados/E19_o_agora.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E19_o_agora.json gravado | 25 grandezas
